# 15.1 语言能力评估 (Language Evaluation)

评估大语言模型的语言理解和生成能力。

本节涵盖：语言建模指标、理解任务评估、生成任务评估

## 1. 语言建模指标与理解任务评估

**语言建模指标**：
- **困惑度（Perplexity）**：模型对文本的预测能力，PPL = exp(交叉熵损失)
- **Bits per character**：每个字符的平均比特数

**理解任务评估**：
- **MMLU**：57个学科的多选题，测试知识广度
- **HellaSwag**：常识推理完形填空
- **ARC**：科学推理题
- **WinoGrande**：共指消解

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

torch.manual_seed(42)

logits = torch.randn(4, 100)
targets = torch.randint(0, 100, (4,))

ce_loss = F.cross_entropy(logits, targets)
perplexity = math.exp(ce_loss.item())

top1 = (logits.argmax(dim=-1) == targets).float().mean()
top5 = sum(targets[i] in logits[i].topk(5).indices for i in range(4)) / 4

print('=== Language Modeling Metrics ===')
print(f'Cross-entropy loss: {ce_loss.item():.4f}')
print(f'Perplexity: {perplexity:.2f}')
print(f'  (Lower PPL = better prediction)')
print(f'\n=== Understanding Task Metrics ===')
print(f'Top-1 accuracy: {top1:.2%}')
print(f'Top-5 accuracy: {top5:.2%}')
print(f'\nKey benchmarks:')
print(f'  MMLU: 57 subjects, tests knowledge breadth')
print(f'  HellaSwag: commonsense reasoning')
print(f'  ARC: scientific reasoning')
print(f'\nKey: Perplexity measures language modeling quality.')
print(f'Benchmark scores measure specific capabilities.')

## 2. 生成任务评估

**自动评估指标**：
- **BLEU**：n-gram精确率，适合翻译
- **ROUGE**：n-gram召回率，适合摘要
- **BERTScore**：基于BERT的语义相似度

**LLM-as-Judge**：用强模型评估弱模型的输出质量。

In [ ]:
import torch

torch.manual_seed(42)

def compute_bleu(reference, hypothesis, max_n=4):
    ref_tokens = reference.split()
    hyp_tokens = hypothesis.split()
    precisions = []
    for n in range(1, max_n + 1):
        ref_ngrams = [tuple(ref_tokens[i:i+n]) for i in range(len(ref_tokens)-n+1)]
        hyp_ngrams = [tuple(hyp_tokens[i:i+n]) for i in range(len(hyp_tokens)-n+1)]
        if not hyp_ngrams:
            precisions.append(0)
            continue
        matches = sum(1 for ng in hyp_ngrams if ng in ref_ngrams)
        precisions.append(matches / len(hyp_ngrams))
    avg_precision = sum(precisions) / len(precisions) if precisions else 0
    bp = min(1, len(hyp_tokens) / max(1, len(ref_tokens)))
    return bp * avg_precision

ref = 'the cat sat on the mat'
hyp_good = 'the cat sat on the mat'
hyp_ok = 'the cat is on the mat'
hyp_bad = 'a dog ran in the park'

print('=== Generation Evaluation ===')
print(f'Reference: "{ref}"')
print(f'\nBLEU scores:')
print(f'  Perfect match: "{hyp_good}" -> BLEU={compute_bleu(ref, hyp_good):.4f}')
print(f'  Partial match: "{hyp_ok}" -> BLEU={compute_bleu(ref, hyp_ok):.4f}')
print(f'  No match:      "{hyp_bad}" -> BLEU={compute_bleu(ref, hyp_bad):.4f}')
print(f'\nKey: BLEU measures n-gram precision (translation).')
print(f'ROUGE measures n-gram recall (summarization).')
print(f'BERTScore measures semantic similarity (more robust).')